# MàthemaSharp.Quantum — Usage

Stati Complex<T>, operatori densi e sistemi finiti. Prerequisiti: build Release, JupyterNet su .NET 11. Constraint: IEEE + IMinMaxValue per il componente reale T.

## Contratti
- Normalize accetta uno stato finito non nullo; scala le componenti per evitare overflow prematuro. Norm può essere infinito se la norma supera il range di T.
- InnerProduct coniuga il primo argomento; ExpectationValue conserva la parte immaginaria e non normalizza automaticamente lo stato.
- CheckEigenstate usa un residuo relativo alla norma Frobenius dell'operatore; tolleranza finita non negativa. È valido anche per operatori non hermitiani.
- DiagonalizeHermitian2 richiede una matrice hermitiana finita 2×2; fasi degli autovettori arbitrarie. Non è una diagonalizzazione generale.
- MatrixExp è Taylor con scaling-and-squaring (24 termini default), non Padé e non un certificatore di accuratezza. TimeEvolution richiede t finito e hbar positivo finito.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System.Numerics;
using System.Runtime.InteropServices;
using MathemaSharp.Quantum;
Console.WriteLine(RuntimeInformation.FrameworkDescription);


.NET 11.0.0-rc.3.26522.148

## Normalizzazione scalata e autostato di X
Il ket [MaxValue,MaxValue] è finito, ma la sua norma supera il range double. Normalize lavora sul ket scalato.


In [ ]:
Complex<double>[] huge = [new(double.MaxValue,0),new(double.MaxValue,0)];
if(!double.IsPositiveInfinity(QuantumAlgorithms.Norm<double>(huge))) throw new Exception("Expected overflowing norm");
var unit = QuantumAlgorithms.Normalize<double>(huge);
if(Math.Abs(QuantumAlgorithms.Norm<double>(unit)-1)>1e-12) throw new Exception("Normalization");
var eigen = QuantumAlgorithms.CheckEigenstate<double>(Pauli<double>.X,unit,1e-12);
if(!eigen.IsEigenstate || (eigen.Eigenvalue-Complex<double>.One).GetMagnitude()>1e-12) throw new Exception("Eigenstate");
if(QuantumAlgorithms.CheckEigenstate<double>(Pauli<double>.X,[Complex<double>.One,Complex<double>.Zero],1e-12).IsEigenstate) throw new Exception("False eigenstate");
Console.WriteLine("normalization: OK");
Console.WriteLine("eigenstate_and_rejection: OK");


normalization: OK
eigenstate_and_rejection: OK

## Diagonalizzazione e matrice densità
Gli autovettori di Y sono ortogonali; uno stato normalizzato produce una matrice densità di traccia uno e purezza uno.


In [ ]:
var spectral = QuantumAlgorithms.DiagonalizeHermitian2(Pauli<double>.Y);
if(Math.Abs(spectral.LambdaPlus-1)>1e-12 || Math.Abs(spectral.LambdaMinus+1)>1e-12) throw new Exception("Eigenvalues");
if(QuantumAlgorithms.InnerProduct<double>(spectral.VectorPlus,spectral.VectorMinus).GetMagnitude()>1e-12) throw new Exception("Orthogonality");
var rho = QuantumAlgorithms.DensityMatrix<double>(spectral.VectorPlus);
if((rho.Trace()-Complex<double>.One).GetMagnitude()>1e-12 || Math.Abs(QuantumAlgorithms.Purity(rho)-1)>1e-12) throw new Exception("Density");
Console.WriteLine("diagonalization: OK");
Console.WriteLine("density_trace_and_purity: OK");


diagonalization: OK
density_trace_and_purity: OK

## Evoluzione temporale Decimal128
Per H=X e t=π/2: exp(-iXt)|0⟩ = -i|1⟩. I confronti sono in Decimal128, non riconvertiti a double.


In [ ]:
var one = Decimal128.One;
var two = Decimal128.CreateChecked(2);
var tiny = one;
for(var i=0;i<28;i++) tiny/=Decimal128.CreateChecked(10);
var evolution = QuantumAlgorithms.TimeEvolution(Pauli<Decimal128>.X,Decimal128.Pi/two);
var ket = QuantumAlgorithms.Apply(evolution,QubitStates<Decimal128>.Zero);
if(ket[0].GetMagnitude()>tiny || (ket[1]+Complex<Decimal128>.ImaginaryOne).GetMagnitude()>tiny) throw new Exception("Evolution Decimal128");
if(!evolution.IsUnitary()) throw new Exception("Unitarity");
Console.WriteLine("evolution_decimal128_1e28: OK");


evolution_decimal128_1e28: OK

## Reference e limiti
[Catalogo API pubblico](12-api-reference.ipynb) · [Verifiche sui sei formati](11-consolidation.ipynb) · [Inventario](../../ARCHITECTURE.md).

L'errore sotto 1e-28 è verificato per questo caso specifico, non per ogni matrice. Spettri sbilanciati, operatori non normali e scaling estremo richiedono analisi separate. ComplexMatrix è una struct con backing array condiviso: assegnare una matrice non crea una copia indipendente dei dati. Nessuna simulazione quantistica infinita o GPU è inclusa.
